Lê Hoàng Phúc

In [11]:
import joblib
import pandas as pd
from utils import *
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

### Artifacts

In [7]:
xg_model = joblib.load("../../../artifacts/xgb_model.joblib")
xgb_features = joblib.load("../../../artifacts/xgb_features.joblib")

lgbm_model = joblib.load("../../../artifacts/lgbm_pipeline.joblib")   
lgbm_features = joblib.load("../../../artifacts/lgbm_features.joblib")

In [9]:
print("XGB num features:", len(xgb_features))
print("LGBM num features:", len(lgbm_features))
print("XGB first 5:", xgb_features)
print("LGBM first 5:", lgbm_features)

XGB num features: 36
LGBM num features: 33
XGB first 5: ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', '_log_amount', 'is_night_proxy', 'is_business_hours_proxy', 'hour_sin', 'hour_cos', 'time_diff', 'is_high_amount', 'is_rapid_transaction']
LGBM first 5: ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', '_log_amount', 'Hour_from_start_mod24', 'is_night_proxy', 'is_business_hours_proxy']


In [12]:
df = pd.read_csv('../../../data/creditcard.csv')
df = create_features(df)
df['_log_amount_raw'] = df['_log_amount']
scaler = StandardScaler()
df[['_log_amount']] = scaler.fit_transform(df[['_log_amount']])
df['hour_sin'] = np.sin(2 * np.pi * df['Hour_from_start_mod24']/24)
df['hour_cos'] = np.cos(2 * np.pi * df['Hour_from_start_mod24']/24)
df['time_diff'] = df['Time'].diff().fillna(0)
threshold = df['Amount'].quantile(0.95)  
df['is_high_amount'] = (df['Amount'] > threshold).astype(int)
df['is_rapid_transaction'] = (df['time_diff'] < 60).astype(int)
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,_log_amount,Hour_from_start_mod24,is_night_proxy,is_business_hours_proxy,_log_amount_raw,hour_sin,hour_cos,time_diff,is_high_amount,is_rapid_transaction
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,1.123062,0,1,0,5.014760,0.0,1.0,0.0,0,1
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-1.115298,0,1,0,1.305626,0.0,1.0,0.0,0,1
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,1.680981,0,1,0,5.939276,0.0,1.0,1.0,1,1
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,1.008128,0,1,0,4.824306,0.0,1.0,0.0,0,1
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,0.669117,0,1,0,4.262539,0.0,1.0,1.0,0,1


### Chưa dùng model artifacts

In [15]:
df = pd.read_csv('../../../data/creditcard.csv')
df = create_features(df)
df['hour_sin'] = np.sin(2 * np.pi * df['Hour_from_start_mod24']/24)
df['hour_cos'] = np.cos(2 * np.pi * df['Hour_from_start_mod24']/24)
df['time_diff'] = df['Time'].diff().fillna(0)
threshold = df['Amount'].quantile(0.95)  
df['is_high_amount'] = (df['Amount'] > threshold).astype(int)
df['is_rapid_transaction'] = (df['time_diff'] < 60).astype(int)
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,Class,_log_amount,Hour_from_start_mod24,is_night_proxy,is_business_hours_proxy,hour_sin,hour_cos,time_diff,is_high_amount,is_rapid_transaction
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,0,5.014760,0,1,0,0.0,1.0,0.0,0,1
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,0,1.305626,0,1,0,0.0,1.0,0.0,0,1
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0,5.939276,0,1,0,0.0,1.0,1.0,1,1
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,0,4.824306,0,1,0,0.0,1.0,0.0,0,1
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,0,4.262539,0,1,0,0.0,1.0,1.0,0,1


In [16]:
features = df.drop(['Time','Class','Amount','Hour_from_start_mod24'], axis=1).columns.tolist()
target = "Class"
X_train, y_train, X_val, y_val, X_test, y_test = split_data(df, features, target)

X_train: (181584, 36) y_train: (181584,)
X_val: (45396, 36) y_val: (45396,)
X_test: (56746, 36) y_test: (56746,)
Fraud rate in train: 0.001910961318177813
Fraud rate in test: 0.0013040566735981391


In [17]:
X_full = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
y_full = pd.concat([y_train, y_val], axis=0).reset_index(drop=True)

In [18]:
pos, neg = int((y_train==1).sum()), int((y_train==0).sum())
xg_model = XGBClassifier(
    n_estimators=800,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.6,
    colsample_bytree=0.9,
    reg_lambda = 1.0,
    gamma=1.0,
    min_child_weight=4,
    tree_method = "hist",
    scale_pos_weight=neg/max(pos, 1),
    eval_metric='aucpr',
    random_state=SEED
)

xg_model.fit(X_train, y_train)

p_val_xg  = xg_model.predict_proba(X_val)[:, 1]

In [19]:
lgb_model = LGBMClassifier(
    n_estimators=800,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=20,
    max_depth=-1,
    lambda_l2=1.0,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=SEED,
    class_weight="balanced",
    eval_metric="average_precision"
)

lgb_model.fit(X_train, y_train)

p_val_lgb  = lgb_model.predict_proba(X_val)[:, 1]

[LightGBM] [Warning] Unknown parameter: eval_metric
[LightGBM] [Warning] lambda_l2 is set=1.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1.0
[LightGBM] [Warning] Unknown parameter: eval_metric
[LightGBM] [Warning] lambda_l2 is set=1.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1.0
[LightGBM] [Info] Number of positive: 347, number of negative: 181237
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.014848 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7467
[LightGBM] [Info] Number of data points in the train set: 181584, number of used features: 35
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] Unknown parameter: eval_metric
[LightGBM] [Warning] lambda_l2 is set=1.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1.0


In [20]:
p_val_blend = 0.6 * p_val_xg + 0.4 * p_val_lgb

In [21]:
rs = []

rs.append({
    "model": "XGB val",
    **log_eval(y_val, p_val_xg)  
})

rs.append({
    "model": "LGB val",
    **log_eval(y_val, p_val_lgb)  
})

rs.append({
    "model": "XGB + LGB val",
    **log_eval(y_val, p_val_blend)  
})

val_df = pd.DataFrame(rs)
val_df

,model,threshold,Cost,ROC_AUC,PR_AUC,debiased_ece,adaptive_ece,Brier
0,XGB val,0.028,2105.0,0.982716,0.794442,0.000378,0.000325,0.000494
1,LGB val,0.132,2275.0,0.979742,0.792812,0.000430,0.000162,0.000449
2,XGB + LGB val,0.017,2140.0,0.983095,0.797056,0.000402,0.000242,0.000459


In [22]:
eval = []

for n in np.linspace(0,1,11):

    eval.append({
        "model": "XGB + LGB val",
        **evaluate(y_val, p_val_blend, thr = n) 
    })

eval_df = pd.DataFrame(eval)
eval_df

,model,threshold,precision,recall,f1,roc_auc,auprc,brier,tp,fp,fn,tn
0,XGB + LGB val,0.0,0.001145,1.000000,0.002288,0.983095,0.797056,0.000459,52,45344,0,0
1,XGB + LGB val,0.1,0.594203,0.788462,0.677686,0.983095,0.797056,0.000459,41,28,11,45316
2,XGB + LGB val,0.2,0.661290,0.788462,0.719298,0.983095,0.797056,0.000459,41,21,11,45323
3,XGB + LGB val,0.3,0.677966,0.769231,0.720721,0.983095,0.797056,0.000459,40,19,12,45325
4,XGB + LGB val,0.4,0.709091,0.750000,0.728972,0.983095,0.797056,0.000459,39,16,13,45328
5,XGB + LGB val,0.5,0.750000,0.750000,0.750000,0.983095,0.797056,0.000459,39,13,13,45331
6,XGB + LGB val,0.6,0.795918,0.750000,0.772277,0.983095,0.797056,0.000459,39,10,13,45334
7,XGB + LGB val,0.7,0.829787,0.750000,0.787879,0.983095,0.797056,0.000459,39,8,13,45336
8,XGB + LGB val,0.8,0.906977,0.750000,0.821053,0.983095,0.797056,0.000459,39,4,13,45340
9,XGB + LGB val,0.9,0.950000,0.730769,0.826087,0.983095,0.797056,0.000459,38,2,14,45342


In [33]:
best_thr = 0.8

In [32]:
xg_model.fit(X_full, y_full)
lgb_model.fit(X_full, y_full)

p_test_xg = xg_model.predict_proba(X_test)[:, 1]
p_test_lgb = lgb_model.predict_proba(X_test)[:, 1]

p_test_blend = 0.6 * p_test_xg + 0.4 * p_test_lgb

[LightGBM] [Warning] Unknown parameter: eval_metric
[LightGBM] [Warning] lambda_l2 is set=1.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1.0
[LightGBM] [Warning] Unknown parameter: eval_metric
[LightGBM] [Warning] lambda_l2 is set=1.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1.0
[LightGBM] [Info] Number of positive: 399, number of negative: 226581
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.019274 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7467
[LightGBM] [Info] Number of data points in the train set: 226980, number of used features: 35
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] Unknown parameter: eval_metric
[LightGBM] [Warning] lambda_l2 is set=1.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1.0


In [34]:
df_test_blend = pd.DataFrame([log_eval(y_test,p_test_blend, best_thr)])
df_test_blend

,threshold,Cost,Precision,Recall,F1,ROC_AUC,PR_AUC,debiased_ece,adaptive_ece,Brier
0,0.8,3625.0,0.918033,0.756757,0.82963,0.983252,0.804714,0.000362,0.000131,0.000433


In [35]:
df_test_blend = pd.DataFrame([evaluate(y_test,p_test_blend, best_thr)])
df_test_blend

,threshold,precision,recall,f1,roc_auc,auprc,brier,tp,fp,fn,tn
0,0.8,0.918033,0.756757,0.82963,0.983252,0.804714,0.000433,56,5,18,56667
